# Tesla Stock Price Prediction Using LSTM

This notebook recreates the workflow shown in the reference screenshots: importing libraries, reading and exploring `Tesla.csv`, visualizing stock data, preprocessing the `Open` price series, creating 60-day sequences, training an LSTM model, evaluating predictions, plotting actual versus predicted prices, and saving the trained model.

**Dataset:** `Tesla.csv` with a `Date` column and stock columns such as `Open`, `High`, `Low`, `Close`, `Volume`, and `Adj Close`.

> In Google Colab, upload `Tesla.csv` when prompted by the first data-loading cell. The model uses historical data for educational demonstration only; it is not financial advice.

## 1. Importing Necessary Libraries

In [ ]:
import math
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Dense, LSTM, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)

## 2. Reading the Data

In [ ]:
# Google Colab: upload Tesla.csv if it is not already in the current directory.
if not os.path.exists("Tesla.csv"):
    try:
        from google.colab import files
        uploaded = files.upload()
    except ImportError:
        print("Place Tesla.csv in the same folder as this notebook.")

df = pd.read_csv("Tesla.csv", parse_dates=["Date"], index_col="Date")
df.head()

In [ ]:
print("Dataset shape:", df.shape)
display(df.describe())
df.info()

In [ ]:
# Check for missing values
df.isna().any()

## 3. Data Visualization

In [ ]:
plt.figure(figsize=(15, 7))
sns.heatmap(df.corr(numeric_only=True), cbar=True, annot=True, cmap="Blues")
plt.title("Correlation Between Tesla Stock Features")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16, 8))
plt.plot(df.index, df["Open"], color="red")
plt.title("Open Tesla Stock Price History")
plt.xlabel("Date", fontsize=14)
plt.ylabel("Open Price (USD)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution / frequency of opening prices, shown as a histogram for readability.
plt.figure(figsize=(10, 5))
plt.hist(df["Open"].dropna(), bins=30)
plt.title("Distribution of Tesla Open Share Prices")
plt.xlabel("Open Share Price (USD)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# Sort chronologically and use the Open price as the prediction column.
df = df.sort_index()
dataset = df[["Open"]].dropna().copy()
data = dataset.values.astype("float32")

print("Prediction data shape:", data.shape)

In [ ]:
# Normalize values to the range [0, 1].
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

### Splitting the Data

Use the first 75% of observations for training and the final 25% for testing. Keeping the time order avoids mixing future observations into the training set.

In [ ]:
train_size = int(len(scaled_data) * 0.75)
train_data = scaled_data[:train_size]
test_data = scaled_data[train_size:]

print("Train size:", len(train_data), "Test size:", len(test_data))

## 5. Creating the Training Set

In [ ]:
# Use the previous 60 trading-day observations to predict the next day's Open price.
time_steps = 60
x_train, y_train = [], []

for i in range(time_steps, len(train_data)):
    x_train.append(train_data[i-time_steps:i, 0])
    y_train.append(train_data[i, 0])

x_train = np.array(x_train, dtype="float32")
y_train = np.array(y_train, dtype="float32")

# LSTM input shape: (samples, time steps, features)
x_train = x_train.reshape((x_train.shape[0], x_train.shape[1], 1))

print("X train shape:", x_train.shape)
print("Y train shape:", y_train.shape)

## 6. Building the LSTM Model

In [ ]:
model = Sequential([
    Input(shape=(x_train.shape[1], 1)),
    LSTM(50, return_sequences=True),
    LSTM(64, return_sequences=False),
    Dense(32),
    Dense(16),
    Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mean_squared_error",
    metrics=["mean_squared_error"]
)

model.summary()

## 7. Fitting the LSTM to the Training Set

In [ ]:
callbacks = [
    EarlyStopping(
        monitor="loss",
        patience=10,
        restore_best_weights=True
    )
]

history = model.fit(
    x_train,
    y_train,
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

## 8. Visualizing Performance

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(history.history["loss"], label="Loss")
if "mean_squared_error" in history.history:
    plt.plot(history.history["mean_squared_error"], label="Mean Squared Error")
plt.title("Training Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Creating the Testing Set

In [ ]:
# Add the 60 observations immediately before the test period so the first test
# prediction has the required historical context.
test_inputs = scaled_data[train_size - time_steps:]
x_test, y_test = [], []

for i in range(time_steps, len(test_inputs)):
    x_test.append(test_inputs[i-time_steps:i, 0])
    y_test.append(test_inputs[i, 0])

x_test = np.array(x_test, dtype="float32").reshape(-1, time_steps, 1)
y_test = np.array(y_test, dtype="float32")

print("X test shape:", x_test.shape)
print("Y test shape:", y_test.shape)

## 10. Predicting the Testing Data

In [ ]:
predictions_scaled = model.predict(x_test, verbose=1)
predictions = scaler.inverse_transform(predictions_scaled)
y_test_prices = scaler.inverse_transform(y_test.reshape(-1, 1))

rmse = math.sqrt(mean_squared_error(y_test_prices, predictions))
mae = mean_absolute_error(y_test_prices, predictions)

print(f"Root Mean Squared Error (RMSE): ${rmse:.2f}")
print(f"Mean Absolute Error (MAE): ${mae:.2f}")

## 11. Visualizing Predictions with the Data

In [ ]:
train = dataset.iloc[:train_size]
test = dataset.iloc[train_size:].copy()
test["Predictions"] = predictions.flatten()

plt.figure(figsize=(16, 6))
plt.title("Tesla Open Stock Price Prediction", fontsize=18)
plt.xlabel("Date", fontsize=14)
plt.ylabel("Open Price (USD)", fontsize=14)
plt.plot(train.index, train["Open"], linewidth=2, label="Train")
plt.plot(test.index, test["Open"], linewidth=2, label="Test")
plt.plot(test.index, test["Predictions"], linewidth=2, label="Predictions")
plt.legend()
plt.tight_layout()
plt.show()

## 12. Save and Reload the Model

In [ ]:
# Save in the recommended Keras format.
model.save("tesla_stock_model.keras")
print("Saved model to tesla_stock_model.keras")

In [ ]:
# Optional: verify that the saved model can be loaded.
loaded_model = load_model("tesla_stock_model.keras")
loaded_model.summary()

## Conclusion

The notebook loads historical Tesla stock data, explores the dataset, trains an LSTM model on the `Open` price using 60 prior observations, reports RMSE and MAE, and plots the predicted values against the held-out test period.

**Important:** This is a learning project. Stock prices are affected by many factors not represented in this univariate historical-price model, so predictions should not be used alone for investment decisions.